# Olist delivery models
**Before running:** put the 9 Olist CSV files in one folder and set `DATA_DIR` in the second cell. Then use *Run All*.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.ensemble import HistGradientBoostingClassifier as HC, HistGradientBoostingRegressor as HR
from sklearn.metrics import (roc_auc_score, average_precision_score, mean_absolute_error,
                             precision_score, recall_score, r2_score)

In [ ]:
DATA_DIR = "."   # <-- change to your folder, e.g. "data" or "/workspaces/yourrepo/olist"
def f(name): return os.path.join(DATA_DIR, name)

## 1. Load

In [ ]:
dates = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
         'order_delivered_customer_date','order_estimated_delivery_date']
o  = pd.read_csv(f('olist_orders_dataset.csv'), parse_dates=dates)
it = pd.read_csv(f('olist_order_items_dataset.csv'))
p  = pd.read_csv(f('olist_products_dataset.csv'))
tr_ = pd.read_csv(f('product_category_name_translation.csv'))
c  = pd.read_csv(f('olist_customers_dataset.csv'))
s  = pd.read_csv(f('olist_sellers_dataset.csv'))
g  = pd.read_csv(f('olist_geolocation_dataset.csv'))
pay = pd.read_csv(f('olist_order_payments_dataset.csv'))

## 2. Prepare

In [ ]:
geo = g.groupby('geolocation_zip_code_prefix')[['geolocation_lat','geolocation_lng']].median()  # one point per zip
tr_ = pd.concat([tr_, pd.DataFrame({'product_category_name':['pc_gamer','portateis_cozinha_e_preparadores_de_alimentos'],
        'product_category_name_english':['pc_gamer','portable_kitchen_food_preparers']})])
p = p.merge(tr_, how='left', on='product_category_name')
p['category'] = p.product_category_name_english.fillna('unknown')
p['volume_cm3'] = p.product_length_cm * p.product_height_cm * p.product_width_cm
for col in ['product_weight_g','volume_cm3']:
    p.loc[p[col] == 0, col] = np.nan

I = (it.merge(p[['product_id','category','product_weight_g','volume_cm3']], on='product_id', how='left')
       .merge(s, on='seller_id', how='left').merge(o, on='order_id', how='left').merge(c, on='customer_id', how='left'))
I = (I.join(geo.rename(columns={'geolocation_lat':'s_lat','geolocation_lng':'s_lng'}), on='seller_zip_code_prefix')
       .join(geo.rename(columns={'geolocation_lat':'c_lat','geolocation_lng':'c_lng'}), on='customer_zip_code_prefix'))
def haversine(a, b, c_, d):
    a, b, c_, d = map(np.radians, [a, b, c_, d])
    x = np.sin((c_-a)/2)**2 + np.cos(a)*np.cos(c_)*np.sin((d-b)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(x))
I['dist_km'] = haversine(I.s_lat, I.s_lng, I.c_lat, I.c_lng)

pp = pay.groupby('order_id').agg(installments=('payment_installments','max'),
                                 pay_type=('payment_type', lambda x: x.mode().iat[0]))
agg = I.groupby('order_id').agg(n_items=('order_item_id','count'), n_sellers=('seller_id','nunique'),
        price=('price','sum'), freight=('freight_value','sum'), weight=('product_weight_g','sum'),
        volume=('volume_cm3','sum'), dist_km=('dist_km','mean'), seller_state=('seller_state','first'),
        category=('category','first')).reset_index()
O = o.merge(agg, on='order_id', how='left').merge(c, on='customer_id').merge(pp, on='order_id', how='left')
O['days_total'] = (O.order_delivered_customer_date - O.order_purchase_timestamp).dt.total_seconds()/86400
O['est_days']   = (O.order_estimated_delivery_date - O.order_purchase_timestamp).dt.total_seconds()/86400
O['late'] = (O.order_delivered_customer_date > O.order_estimated_delivery_date).astype(float)\
              .where(O.order_delivered_customer_date.notna())

# delivered orders with a known delivery date and distance; drop extreme (>90 day) cases
D = O[(O.order_status=='delivered') & O.order_delivered_customer_date.notna() & O.dist_km.notna()].copy()
D = D[D.days_total <= 90]
t = D.order_purchase_timestamp
D['approve_hrs'] = ((D.order_approved_at - t).dt.total_seconds()/3600).clip(0, 200)
D['month'], D['dow'], D['hour'] = t.dt.month, t.dt.dayofweek, t.dt.hour
D['same_state'] = (D.customer_state == D.seller_state).astype(int)
D['freight_ratio'] = D.freight / D.price.clip(lower=1)
cat = ['customer_state','seller_state','category','pay_type']
for col in cat: D[col] = D[col].astype('category')
num = ['price','freight','freight_ratio','weight','volume','n_items','n_sellers','dist_km','est_days',
       'approve_hrs','month','dow','hour','same_state','installments']
feats = num + cat

## 3. Time-based split

In [ ]:
cut = pd.Timestamp('2018-04-01')
tr, te = D[t < cut], D[t >= cut]
print(f"train {len(tr):,}  test {len(te):,}  late rate train/test {tr.late.mean():.3f} / {te.late.mean():.3f}")

## 4. Model 1: will it be late?

In [ ]:
clf = HC(max_iter=300, learning_rate=0.05, max_leaf_nodes=20, categorical_features='from_dtype',
         class_weight='balanced', random_state=0).fit(tr[feats], tr.late)
pr = clf.predict_proba(te[feats])[:, 1]
print(f"\nLATE MODEL  AUC {roc_auc_score(te.late, pr):.3f}  PR-AUC {average_precision_score(te.late, pr):.3f} "
      f"(guessing = {te.late.mean():.3f})")
for th in [0.5, 0.6, 0.7]:
    fl = pr >= th
    print(f"  alert >= {th}: flagged {fl.mean():.1%}, precision {precision_score(te.late, fl):.1%}, "
          f"recall {recall_score(te.late, fl):.1%}")

## 5. Model 2: how many days?

In [ ]:
reg = HR(max_iter=400, learning_rate=0.05, max_leaf_nodes=20, categorical_features='from_dtype',
         loss='absolute_error', random_state=0).fit(tr[feats], tr.days_total)
pd_ = reg.predict(te[feats])
print(f"\nDAYS MODEL  average miss {mean_absolute_error(te.days_total, pd_):.2f} days, R2 {r2_score(te.days_total, pd_):.3f}")
print(f"  Olist promised date, average miss: {mean_absolute_error(te.days_total, te.est_days):.2f} days")
print(f"  always guess the middle value:     {mean_absolute_error(te.days_total, np.full(len(te), tr.days_total.median())):.2f} days")
print(f"  within 3 days: model {(abs(te.days_total-pd_)<=3).mean():.1%}, Olist promise {(abs(te.days_total-te.est_days)<=3).mean():.1%}")